# Advanced Concepts Of Python

## `is` vs `==`

### The Golden Rule: Only Use `is` for Singletons

Use `is` and `is not` **exclusively** when comparing against built-in singletons (objects that exist as a single instance in memory throughout the program's lifetime):

### Never Use `is` for Numbers, Strings, or Collections

Never use `is` to compare integers, floats, strings, lists, or dicts even if it _seems_ to work in small tests:


In [ ]:
class Fruit:
    def __init__(self, name: str, calories: float):
        self.name = name
        self.calories = calories

    def __eq__(self, other):
        return self.__dict__ == other.__dict__


if __name__ == '__main__':
    fruit_a: Fruit = Fruit('Banana', 10)
    fruit_b: Fruit = Fruit('Banana', 10)

    print(f'fruit_a is fruit_b = {fruit_a is fruit_b}')
    # False (Nor in the same memory)
    print(f'fruit_a == fruit_b = {fruit_a == fruit_b}')
    # True
    # (Still not in the same memory
    # the object dicts have the same values and keys)

fruit_a is fruit_b = False
fruit_a == fruit_b = True


## Lambdas

Could mess up the readability in usage of complex scenarios or deep logics.


In [ ]:
a = 2
sq = lambda a: a**2
print(sq(a))

4


## Walrus Operator :=

the **walrus operator (`:=`)**—officially called an **assignment expression** (introduced in Python 3.8)—lets you assign a value to a variable **and return that value at the same time** inside an expression.

It gets its nickname because `:=` looks like a walrus lying on its side (eyes and tusks).

### Why It Exists (Eliminating Duplicate Work)

Normally, Python's `=` is a _statement_, meaning you cannot put it inside an `if`, `while`, or list comprehension. Before `:=`, you either had to write an extra line or call the same function twice.


In [ ]:
data = [10, 20, 30, 40, 50, 60]

if len(data) > 5:
    print(f'Too many items: {len(data)}')

if (n := len(data)) > 5:
    print(f'Too many items: {n}')

### Most Practical Use Cases

#### Cleaner `while` Loops (Reading Streams / User Input)

Instead of writing an infinite `while True:` loop with a `break`, you can assign and check the condition in the `while` header:


In [ ]:
# Keeps reading user input until they type "quit"
while (command := input('Enter command: ')) != 'quit':
    print(f'Running: {command}')

#### List Comprehensions (Avoid Calling Expensive Functions Twice)

When filtering and transforming data in a comprehension, `:=` prevents calling a function once in `if` and a second time in the output:


In [ ]:
def slow_process(x: int) -> int:
    return x * 10


numbers = [1, 2, 3, 4, 5]

# Calls slow_process(x) only ONCE per item instead of twice
results = [val for x in numbers if (val := slow_process(x)) > 20]
print(results)  # [30, 40, 50]

[30, 40, 50]


#### Regex Matching

Check if a pattern matched and immediately use the match object inside the `if` block:


In [ ]:
import re

text = 'Order ID: 9482'

# Assigns the Match object to `match` and checks if it is not None
if (match := re.search(r'\d+', text)) is not None:
    print(f'Found ID: {match.group()}')  # Found ID: 9482

### Key Rules & Gotchas

- **Parentheses matter:** Because `:=` has very low operator precedence (lower than `>`, `==`, `!=`, and `is`), always wrap `(var := expr)` in parentheses when comparing its result—otherwise `if n := len(data) > 5:` assigns `True` (a boolean) to `n` instead of `6`.

- **Not a replacement for `=`:** Do not use `:=` for standalone assignments like `(x := 10)` on its own line; use standard `x = 10` for readability.


In [ ]:
def get_info(text: str) -> dict:
	return {
		'text': text,
		'letters': (length := len(text.replace(' ', ''))),
		'words': (words := text.split()),
		'total_words': (word_length := len(words)),
		'avg_per_word': round(length / word_length, 2)
	}

### delete


In [ ]:
people: list[str] = ['Mario', 'Toad']

del people[1]

print(people) # ['Mario'] Also works for dict key val pairs

name = 'Mario'

del name

print(name) # Runtime error happens


### assert

`assert` is a built-in debugging statement in Python used to test whether a condition is `True`. If the condition evaluates to `True`, execution continues normally; if it evaluates to `False`, Python immediately halts the program by raising an `AssertionError`.


In [ ]:
def calculate_discount(price: float, discount: float) -> float:
    assert 0.0 <= discount <= 1.0, f'Discount must be between 0 and 1, got {discount}'
    return price * (1.0 - discount)


print(calculate_discount(100.0, 0.2))
# 80.0
print(calculate_discount(100.0, 1.5))
# Raises AssertionError: Discount must be between 0 and 1, got 1.5

#### When to Use `assert` vs. `raise` (Critical Distinction)

`assert` is **not** a substitute for runtime error handling or user input validation, because Python completely strips out all `assert` statements when run with the optimization flag (`python -O script.py` or `PYTHONOPTIMIZE=1`).

| **Use Case**                                                                                              | **Mechanism**                             | **Why**                                                                                           |
| --------------------------------------------------------------------------------------------------------- | ----------------------------------------- | ------------------------------------------------------------------------------------------------- |
| **Internal invariants & developer bugs** (conditions that should _never_ happen if your code is bug-free) | `assert condition, "msg"`                 | Catches logic bugs early during development and testing; zero overhead in `-O` production builds. |
| **User input, API data, file I/O, permissions** (errors that _can_ happen in real usage)                  | `if not condition: raise ValueError(...)` | Always executes even when optimization flags (`-O`) are enabled.                                  |


In [ ]:
# WRONG: Never validate external/user input with assert (skipped under `python -O`)
def withdraw_bad(amount: float) -> None:
    assert amount > 0, 'Amount must be positive'


# RIGHT: Always raise explicit exceptions for runtime validation
def withdraw_good(amount: float) -> None:
    if amount <= 0:
        raise ValueError('Amount must be positive')

### Common Syntax Trap: Never Add Parentheses Around Both Arguments

Writing `assert(condition, "message")` passes a **2-element tuple** `(condition, "message")` to `assert`. Because a non-empty tuple is always truthy in Python, that assertion will **never fail** even when `condition` is `False`:


In [ ]:
# BUG: Non-empty tuple is always True — never raises an error!


assert (1 == 2, 'This will never fail')  # noqa: F631, PLR0133

# NOTE: No outer parentheses

assert 1 == 2, 'This properly raises AssertionError'  # noqa: PLR0133


### Match Case


In [ ]:
status: int = 400

match status:
    case 200:
        print('Success')
    case 400:
        print('Bad Request')
    case 403:
        print('Forbidden')
    case 404:
        print('Not Found')
    case _:
        print('Unknown Status')

### Pattern Matching in Match Case


In [ ]:
user_input: str = 'find image1.png image2.jpg image3.webp'
p_command: list[str] = user_input.split()

match p_command:
    case ['find', *images]:
        print(f'Finding: {", ".join(images)}')
    case ['download', *images]:
        print(f'Finding: {", ".join(images)}')
    case ['cancel' | 'delete', *images] if len(images) > 1:
        print(f'Deleting: {", ".join(images)}')
    case _:
        print('Unknown Command')


Finding image1.png, image2.jpg, image3.webp
